In [1]:
import wrds
import pandas as pd
import numpy as np
import s3fs

In [112]:
db = wrds.Connection(wrds_username='vincent0358')

Loading library list...
Done


In [ ]:
#create_pgpass_file(db)

In [6]:
db.list_libraries()

['aha_sample',
 'ahasamp',
 'audit',
 'audit_acct_os',
 'audit_audit_comp',
 'audit_common',
 'audit_corp_legal',
 'auditsmp',
 'auditsmp_all',
 'bank',
 'bank_all',
 'bank_premium_samp',
 'banksamp',
 'block',
 'block_all',
 'boardex',
 'boardex_eur',
 'boardex_na',
 'boardex_row',
 'boardex_trial',
 'boardex_uk',
 'boardsmp',
 'bvd_amadeus_trial',
 'bvd_bvdbankf_trial',
 'bvd_orbis_trial',
 'bvdsamp',
 'calcbench_trial',
 'calcbnch',
 'candid_samp',
 'cboe',
 'cboe_all',
 'cboe_sample',
 'cboesamp',
 'cddsamp',
 'ciq',
 'ciq_common',
 'ciq_pplintel',
 'ciqsamp',
 'ciqsamp_capstrct',
 'ciqsamp_common',
 'ciqsamp_keydev',
 'ciqsamp_pplintel',
 'ciqsamp_ratings',
 'ciqsamp_transactions',
 'ciqsamp_transcripts',
 'cisdmsmp',
 'columnar',
 'comp',
 'comp_bank_daily',
 'comp_execucomp',
 'comp_global_daily',
 'comp_na_daily_all',
 'comp_segments_hist_daily',
 'comp_snapshot',
 'compsamp',
 'compsamp_all',
 'compsamp_snapshot',
 'compseg',
 'compsnap',
 'contrib',
 'contrib_as_filed_financi

In [8]:
with wrds.Connection() as db:
    stocknames = db.get_table(library='crsp', table='stocknames', rows=10)
stocknames.head()

WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\jiangao\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


,permno,namedt,nameenddt,shrcd,exchcd,siccd,ncusip,ticker,comnam,shrcls,permco,hexcd,cusip,st_date,end_date,namedum
0,10000,1986-01-07,1987-06-11,10,3,3990,68391610,OMFGA,OPTIMUM MANUFACTURING INC,A,7952,3,68391610,1986-01-07,1987-06-11,2.0
1,10001,1986-01-09,1993-11-21,11,3,4920,39040610,GFGC,GREAT FALLS GAS CO,<NA>,7953,2,36720410,1986-01-09,2017-08-03,2.0
2,10001,1993-11-22,2008-02-04,11,3,4920,29274A10,EWST,ENERGY WEST INC,<NA>,7953,2,36720410,1986-01-09,2017-08-03,2.0
3,10001,2008-02-05,2009-08-03,11,3,4920,29274A20,EWST,ENERGY WEST INC,<NA>,7953,2,36720410,1986-01-09,2017-08-03,2.0
4,10001,2009-08-04,2009-12-17,11,3,4920,29269V10,EGAS,ENERGY INC,<NA>,7953,2,36720410,1986-01-09,2017-08-03,2.0


## Deal with company IDs

To use data from WRDS, we link the firm IDs in different systems as follows:
* ISIN to CUSIP:LSEG / LSEG Common Files / Security Master;
* CUSIP to ticker: CRSP/Compustat Merged Database - Linking Table
* To PERMNO: use db.get_table(library='crsp', table='stocknames', rows=10), in which there're PERMNO, CUSIP, and Ticker. Permno can be used for fetching factors in Factors by WRDS.

### 1 Get ISIN for U.S. companies

In [3]:
s3_file = "s3://buc-vin0358/trucost_use.csv"

trucost_all = pd.read_csv(s3_file)

In [4]:
tru_country = trucost_all.groupby('Country').count()

In [5]:
tru_country.head()

,TCUID,Company,ISIN,FinancialYear,GICS_Sector_Code,GICS_Sector_Name,GICS_Industry_Group_Code,GICS_Industry_Group_Name,GICS_Industry_Code,GICS_Industry_Name,...,TotalDirect(USDmn),TotalIndirect(USDmn),Total_DirectAndIndirect(USDmn),GHG_Direct(USDmn),GHG_Indirect(USDmn),GHG_Total(USDmn),GHG_DirectImpactRatio,GHG_IndirectImpactRatio,GHG_TotalImpactRatio,EffectiveDate
Country,,,,,,,,,,,,,,,,,,,,,
ARGENTINA,125,125,125,125,125,125,125,125,125,125,...,125,125,125,125,125,125,125,125,125,125
AUSTRALIA,4429,4429,4412,4429,4429,4429,4429,4429,4429,4429,...,4429,4429,4429,4429,4429,4429,4429,4429,4429,4429
AUSTRIA,410,410,410,410,410,410,410,410,410,410,...,410,410,410,410,410,410,410,410,410,410
BAHAMAS,7,7,7,7,7,7,7,7,7,7,...,7,7,7,7,7,7,7,7,7,7
BAHRAIN,35,35,35,35,35,35,35,35,35,35,...,35,35,35,35,35,35,35,35,35,35


In [9]:
US = trucost_all[trucost_all['Country']== 'UNITED STATES']

In [10]:
US.head()

,TCUID,Company,ISIN,FinancialYear,GICS_Sector_Code,GICS_Sector_Name,GICS_Industry_Group_Code,GICS_Industry_Group_Name,GICS_Industry_Code,GICS_Industry_Name,...,TotalDirect(USDmn),TotalIndirect(USDmn),Total_DirectAndIndirect(USDmn),GHG_Direct(USDmn),GHG_Indirect(USDmn),GHG_Total(USDmn),GHG_DirectImpactRatio,GHG_IndirectImpactRatio,GHG_TotalImpactRatio,EffectiveDate
360,42729,Tyco International PLC,IE00BQRQXQ92,2005,20.0,Industrials,2010.0,Capital Goods,201050.0,Industrial Conglomerates,...,53.388483,527.068493,580.456976,25.413463,240.684383,266.097846,0.063337,0.599847,0.663184,2009-05-15 12:48:46.67
361,42729,Tyco International PLC,IE00BQRQXQ92,2006,20.0,Industrials,2010.0,Capital Goods,201050.0,Industrial Conglomerates,...,53.412588,498.142054,551.554642,25.368609,225.563068,250.931677,0.061935,0.550691,0.612626,2009-05-15 12:48:46.67
362,42729,Tyco International PLC,IE00BQRQXQ92,2007,20.0,Industrials,2010.0,Capital Goods,201050.0,Industrial Conglomerates,...,24.171680,226.126716,250.298397,12.455727,108.797466,121.253194,0.066321,0.579295,0.645616,2009-05-15 12:48:46.67
363,42729,Tyco International PLC,IE00BQRQXQ92,2008,20.0,Industrials,2010.0,Capital Goods,201050.0,Industrial Conglomerates,...,26.583626,229.062354,255.645980,13.618574,113.313961,126.932535,0.067422,0.560988,0.628410,2011-08-03 12:39:50.92
364,42729,Tyco International PLC,IE00BQRQXQ92,2009,20.0,Industrials,2010.0,Capital Goods,201050.0,Industrial Conglomerates,...,21.645412,200.234121,221.879533,13.918476,95.823620,109.742096,0.080748,0.555918,0.636666,2012-02-08 05:27:43.42


In [11]:
len(US)

23741

In [12]:
file = "s3://buc-vin0358/US_trucost.csv"
US.to_csv(file)

In [13]:
unique_isin = US['ISIN'].unique()

file = "C:/Users/jiangao/QuantProjects/CEO_SC/data/US_isin.txt"

with open(file, 'w') as f:
    for isin in unique_isin:
        f.write(f"{isin}\n")
    f.close()

In [15]:
file = "C:/Users/jiangao/QuantProjects/CEO_SC/data/US_isin_cusip.csv"
isin_cusip = pd.read_csv(file)

isin_cusip = isin_cusip.dropna(subset=['cusip'], how='all')
isin_cusip = isin_cusip.drop_duplicates(subset=['cusip'], keep='first')

In [16]:
len(isin_cusip)

3491

In [19]:
stocknames = db.get_table(library='crsp', table='stocknames')

In [20]:
len(stocknames)

83280

In [21]:
stocknames.columns

Index(['permno', 'namedt', 'nameenddt', 'shrcd', 'exchcd', 'siccd', 'ncusip',
       'ticker', 'comnam', 'shrcls', 'permco', 'hexcd', 'cusip', 'st_date',
       'end_date', 'namedum'],
      dtype='object')

In [22]:
isin_cusip = isin_cusip.merge(stocknames, how='left', on='cusip')

In [24]:
len(isin_cusip)

10595

In [25]:
file = "s3://buc-vin0358/US_isin_cusip_permno_ticker.csv"
isin_cusip.to_csv(file)

In [2]:
file = "s3://buc-vin0358/US_isin_cusip_permno_ticker.csv"
isin_cusip = pd.read_csv(file)

In [3]:
isin_cusip.head()

,Unnamed: 0,isin,cusip,permno,namedt,nameenddt,shrcd,exchcd,siccd,ncusip,ticker,comnam,shrcls,permco,hexcd,st_date,end_date,namedum
0,0,US0003602069,00036020,76868.0,1991-01-03,1993-09-15,10.0,3.0,3580.0,00036010,AAON,AAON INC,NaN,10817.0,3.0,1991-01-03,2024-12-31,2.0
1,1,US0003602069,00036020,76868.0,1993-09-16,2024-12-31,11.0,3.0,3580.0,00036020,AAON,AAON INC,NaN,10817.0,3.0,1991-01-03,2024-12-31,2.0
2,2,US0003611052,00036110,54594.0,1972-04-24,1980-11-16,11.0,2.0,3662.0,00036110,AIR,A A R CORP,NaN,20000.0,1.0,1972-04-24,2024-12-31,2.0
3,3,US0003611052,00036110,54594.0,1980-11-17,1997-02-02,11.0,1.0,3662.0,00036110,AIR,A A R CORP,NaN,20000.0,1.0,1972-04-24,2024-12-31,2.0
4,4,US0003611052,00036110,54594.0,1997-02-03,2006-05-24,11.0,1.0,5088.0,00036110,AIR,A A R CORP,NaN,20000.0,1.0,1972-04-24,2024-12-31,2.0


In [4]:
len(isin_cusip)

10595

In [5]:
from dateutil.relativedelta import relativedelta

In [29]:
def expand_dates(df, start_col = 'st_date', end_col = 'end_date', date_format = None):

    df_copy = df.copy()

    # convert date column if it's not already datetime
    #if not pd.api.types.is_datetime64_dtype(df_copy[start_col]):
    df_copy[start_col] = pd.to_datetime(df_copy[start_col], format=date_format, errors='coerce')

    #if not pd.api.types.is_datetime64_dtype(df_copy[end_col]):
    df_copy[end_col] = pd.to_datetime(df_copy[end_col], format=date_format, errors='coerce')
        
    # empty list to store expanded rows
    expanded_rows = []

    # iterate
    for _, row in df.iterrows():
        # get start and end dates
        start = row[start_col]
        end = row[end_col]

        if pd.isna(start) or pd.isna(end):
            continue

        start = pd.to_datetime(start)
        end = pd.to_datetime(end)

        current = start
        
        # specify another start considering sample availability
        sample_start = pd.to_datetime('2005-01-01')
        while current <= end and current >= sample_start:
            new_row = row.copy()
            new_row['date'] = current
            new_row['year'] = current.year
            new_row['month'] = current.month
            expanded_rows.append(new_row)

            # move to next month
            current += relativedelta(months=1)

    expanded_df = pd.DataFrame(expanded_rows)

    return expanded_df

In [10]:
isin_cusip_use = isin_cusip.dropna(subset=['permno'], how='all')
isin_cusip_use = isin_cusip_use.drop_duplicates(subset=['isin', 'permno', 'st_date', 'end_date'], keep='first')
isin_cusip_use['permno'] = isin_cusip_use['permno'].astype(int)

In [30]:
isin_cusip_expand = expand_dates(isin_cusip_use, start_col = 'st_date', end_col = 'end_date')

In [31]:
file = "s3://buc-vin0358/US_isin_cusip_permno_ticker_expand_temp.csv"
isin_cusip_expand.to_csv(file)

In [25]:
isin_cusip_expand['cusip'].isna().sum()

np.int64(0)

In [32]:
# drop duplicates
isin_cusip_expand = isin_cusip_expand.sort_values(by=['isin', 'permno', 'year', 'month'], ascending=True)
isin_cusip_expand = isin_cusip_expand.drop_duplicates(subset=['isin', 'permno', 'year', 'month'], keep='first')

In [34]:
len(isin_cusip_expand)

174300

In [35]:
file = "s3://buc-vin0358/US_isin_cusip_permno_ticker_expand.csv"
isin_cusip_expand.to_csv(file)

In [60]:
isin_cusip_expand.columns

Index(['Unnamed: 0', 'isin', 'cusip', 'permno', 'namedt', 'nameenddt', 'shrcd',
       'exchcd', 'siccd', 'ncusip', 'ticker', 'comnam', 'shrcls', 'permco',
       'hexcd', 'st_date', 'end_date', 'namedum', 'date', 'year', 'month'],
      dtype='object')

In [69]:
unique_id = isin_cusip_expand['permno'].unique()
unique_id_df = pd.DataFrame(unique_id)
unique_id_df.columns = ['permno']

In [37]:
permno_tuple = tuple(unique_id_df)

In [119]:
permno_list = ','.join(str(key) for key in unique_id_df['permno'])

In [56]:
query = f"""
SELECT permno, mthcaldt, mthprc, mthret, fyear, at
FROM comp.funda
WHERE permno IN ({permno_list})
AND fyear>=2005
"""

In [90]:
temp = db.get_table(library='crsp_a_stock', table='msf', rows=5)
temp

,cusip,permno,permco,issuno,hexcd,hsiccd,date,bidlo,askhi,prc,...,ret,bid,ask,shrout,cfacpr,cfacshr,altprc,spread,altprcdt,retx
0,68391610,10000,7952,10396,3,3990,1985-12-31,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-2.5625,<NA>,1986-01-07,<NA>
1,68391610,10000,7952,10396,3,3990,1986-01-31,-2.5,-4.4375,-4.375,...,<NA>,<NA>,<NA>,3680.0,1.0,1.0,-4.375,0.25,1986-01-31,<NA>
2,68391610,10000,7952,10396,3,3990,1986-02-28,-3.25,-4.375,-3.25,...,-0.257143,<NA>,<NA>,3680.0,1.0,1.0,-3.25,0.25,1986-02-28,-0.257143
3,68391610,10000,7952,10396,3,3990,1986-03-31,-3.25,-4.4375,-4.4375,...,0.365385,<NA>,<NA>,3680.0,1.0,1.0,-4.4375,0.125,1986-03-31,0.365385
4,68391610,10000,7952,10396,3,3990,1986-04-30,-4.0,-4.3125,-4.0,...,-0.098592,<NA>,<NA>,3793.0,1.0,1.0,-4.0,0.25,1986-04-30,-0.098592


In [108]:
temp = db.get_table(library='crspsamp_all', table='crspmsf', rows=10)
temp

,cusip,permno,permco,issuno,hexcd,hsiccd,date,ncusip,ticker,comnam,...,prc2,nmsbid,nmsask,cfacpr,cfacshr,vwretd,vwretx,ewretd,ewretx,sprtrn
0,59491810,10107.0,8048.0,0.0,3.0,7370.0,1986-03-31,59491810,MSFT,MICROSOFT CORP,...,<NA>,27.25,27.5,288.0,288.0,0.053954,0.051431,0.047311,0.0458,0.052794
1,59491810,10107.0,8048.0,0.0,3.0,7370.0,1986-04-30,59491810,MSFT,MICROSOFT CORP,...,<NA>,31.75,32.25,288.0,288.0,-0.007902,-0.009633,0.016128,0.015144,-0.014148
2,59491810,10107.0,8048.0,0.0,3.0,7370.0,1986-05-30,59491810,MSFT,MICROSOFT CORP,...,<NA>,34.75,35.0,288.0,288.0,0.050805,0.045918,0.036189,0.034645,0.050229
3,59491810,10107.0,8048.0,0.0,3.0,7370.0,1986-06-30,59491810,MSFT,MICROSOFT CORP,...,<NA>,30.5,30.75,288.0,288.0,0.01424,0.011619,0.008087,0.006715,0.01411
4,59491810,10107.0,8048.0,0.0,3.0,7370.0,1986-07-31,59491810,MSFT,MICROSOFT CORP,...,<NA>,28.5,28.75,288.0,288.0,-0.059698,-0.061452,-0.073128,-0.073917,-0.058683
5,59491810,10107.0,8048.0,0.0,3.0,7370.0,1986-08-29,59491810,MSFT,MICROSOFT CORP,...,<NA>,28.25,28.5,288.0,288.0,0.06618,0.062427,0.023269,0.02189,0.071193
6,59491810,10107.0,8048.0,0.0,3.0,7370.0,1986-09-30,59491810,MSFT,MICROSOFT CORP,...,<NA>,28.25,28.75,288.0,288.0,-0.079015,-0.081267,-0.059071,-0.060184,-0.085439
7,59491810,10107.0,8048.0,0.0,3.0,7370.0,1986-10-31,59491810,MSFT,MICROSOFT CORP,...,<NA>,38.75,39.25,288.0,288.0,0.04931,0.046866,0.023777,0.022803,0.054729
8,59491810,10107.0,8048.0,0.0,3.0,7370.0,1986-11-28,59491810,MSFT,MICROSOFT CORP,...,<NA>,49.75,50.0,288.0,288.0,0.015079,0.010859,-0.006263,-0.007526,0.021477
9,59491810,10107.0,8048.0,0.0,3.0,7370.0,1986-12-31,59491810,MSFT,MICROSOFT CORP,...,<NA>,48.25,48.5,288.0,288.0,-0.02639,-0.02906,-0.034609,-0.036117,-0.028288


In [118]:
permno_list

'16496, 15053, 14298, 91818, 14235, 12787, 92261, 14704, 19583, 14351, 93340, 13760, 91600, 12076, 14088, 92379, 17131, 14918, 16143, 90940, 21257, 12411, 10201, 91367, 14619, 14336, 14177, 16682, 18904, 91393, 16281, 16383, 18481, 17227, 93296, 14319, 13452, 14784, 15331, 12542, 15702, 92457, 91487, 92096, 92156, 13706, 14011, 16342, 14297, 13586, 12919, 14157, 16711, 17040, 13634, 13789, 93189, 93427, 92041, 16525, 17893, 15913, 14674, 13866, 91655, 14092, 90943, 18426, 14642, 90724, 15597, 15487, 16454, 12345, 14089, 14466, 12587, 16641, 18136, 91065, 13837, 92506, 15170, 15857, 14945, 90825, 12480, 15056, 17223, 13567, 17045, 12828, 92680, 12880, 14532, 93091, 19317, 90548, 14495, 13721, 16051, 13105, 14176, 91814, 14530, 15775, 91119, 16434, 14589, 93073, 91975, 14238, 16276, 14498, 14759, 15829, 16435, 91733, 14044, 14590, 12641, 90716, 16247, 14531, 16814, 16625, 15409, 92032, 16347, 18376, 18941, 13429, 93339, 15774, 91659, 13284, 14558, 18224, 90547, 91348, 16677, 15435, 17034

In [132]:
query_ccm = """
SELECT gvkey,
       lpermno AS permno,
       lpermco AS permco,
       linktype,
       linkprim,
       linkdt,
       linkenddt
FROM crsp.ccmxpf_linktable
WHERE (linktype IN ('LU', 'LC'))
    AND linkprim IN ('P', 'C')
    AND usedflag = 1
"""

query_crsp_m = f"""
SELECT permno, date, ret, shrout, prc
FROM crsp.msf
WHERE permno IN ({permno_list})
    AND date >= '2004-12-31'
"""

query_comp_a = """
SELECT gvkey, datadate, at, ceq, sale, ni
FROM comp.funda
WHERE indfmt = 'INDL'
    AND datadate >= '2004-12-31'
    AND datafmt = 'STD'
    AND popsrc = 'D'
    AND consol = 'C'
"""

In [ ]:
comp = db.raw_sql(query_comp_a)
ccm = db.raw_sql(query_ccm)
crsp_m = db.raw_sql(query_crsp_m)

In [155]:
comp_ccm['gvkey'].isna().sum()

np.int64(0)

In [ ]:
# merge compustat with CCM by gvkey
comp_ccm = comp.merge(ccm, on='gvkey', how='left')

# enforce link date validity
comp_ccm = comp_ccm[
    (comp_ccm['datadate'] >= comp_ccm['linkdt'])&
    ((comp_ccm['datadate'] <= comp_ccm['linkenddt']) | comp_ccm['linkenddt'].isna())
]

comp_ccm = comp_ccm.dropna(subset=['permno'])

# convert permno format as integer
crsp_m['permno'] = crsp_m['permno'].astype(int)
comp_ccm['permno'] = comp_ccm['permno'].astype(int)

In [143]:
# merge with CRSP
merged = comp_ccm.merge(crsp_m, on='permno', how='right')

In [195]:
merged['date_dt'] = pd.to_datetime(merged['date'], errors='coerce')
merged['year'] = merged['date_dt'].dt.year
merged['month'] = merged['date_dt'].dt.month

In [196]:
merged.head()

,gvkey,datadate,at,ceq,sale,ni,permno,permco,linktype,linkprim,linkdt,linkenddt,date,ret,shrout,prc,date_dt,year,month
0,185128,2010-12-31,582.451,195.052,618.227,28.726,10158,53454.0,LC,P,2010-07-22,<NA>,2010-06-30,<NA>,<NA>,<NA>,2010-06-30,2010,6
1,185128,2011-12-31,645.597,236.357,728.2,34.727,10158,53454.0,LC,P,2010-07-22,<NA>,2010-06-30,<NA>,<NA>,<NA>,2010-06-30,2010,6
2,185128,2012-12-31,675.472,261.847,631.171,18.36,10158,53454.0,LC,P,2010-07-22,<NA>,2010-06-30,<NA>,<NA>,<NA>,2010-06-30,2010,6
3,185128,2013-12-31,604.66,276.797,574.171,2.414,10158,53454.0,LC,P,2010-07-22,<NA>,2010-06-30,<NA>,<NA>,<NA>,2010-06-30,2010,6
4,185128,2014-12-31,629.659,286.307,593.241,10.383,10158,53454.0,LC,P,2010-07-22,<NA>,2010-06-30,<NA>,<NA>,<NA>,2010-06-30,2010,6


In [197]:
merged['ret'].isna().sum()

np.int64(38197)

In [198]:
merged_use = merged.dropna(subset=['gvkey', 'year', 'month', 'ret', 'shrout', 'prc', 'at', 'ceq', 'sale', 'ni'])

In [199]:
merged_use = merged_use.drop_duplicates(subset=['gvkey', 'permno', 'year', 'month'], keep='first')

In [200]:
len(merged_use)

177541

In [201]:
merged_use = pd.DataFrame(merged_use)
file = "s3://buc-vin0358/US_wrds_crsp_comp.csv"
merged_use.to_csv(file)

In [202]:
merged_use.columns

Index(['gvkey', 'datadate', 'at', 'ceq', 'sale', 'ni', 'permno', 'permco',
       'linktype', 'linkprim', 'linkdt', 'linkenddt', 'date', 'ret', 'shrout',
       'prc', 'date_dt', 'year', 'month'],
      dtype='object')

In [164]:
isin_cusip_expand.columns

Index(['Unnamed: 0', 'isin', 'cusip', 'permno', 'namedt', 'nameenddt', 'shrcd',
       'exchcd', 'siccd', 'ncusip', 'ticker', 'comnam', 'shrcls', 'permco',
       'hexcd', 'st_date', 'end_date', 'namedum', 'date', 'year', 'month'],
      dtype='object')

In [203]:
temp_merge = isin_cusip_expand.merge(merged_use, left_on=['permno', 'year', 'month'], right_on = ['permno', 'year', 'month'], how='left')

In [204]:
len(temp_merge)

180503

In [205]:
temp_merge.head()

,Unnamed: 0,isin,cusip,permno,namedt,nameenddt,shrcd,exchcd,siccd,ncusip,...,permco_y,linktype,linkprim,linkdt,linkenddt,date_y,ret,shrout,prc,date_dt
0,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,G0684D10,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT
1,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,G0684D10,...,55804.0,LC,P,2016-12-09,2021-12-31,2017-01-31,-0.022088,52901.0,46.93,2017-01-31
2,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,G0684D10,...,55804.0,LC,P,2016-12-09,2021-12-31,2017-02-28,0.107394,52901.0,51.97,2017-02-28
3,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,G0684D10,...,55804.0,LC,P,2016-12-09,2021-12-31,2017-03-31,-0.038099,77410.0,49.99,2017-03-31
4,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,G0684D10,...,55804.0,LC,P,2016-12-09,2021-12-31,2017-04-28,0.066413,101619.0,53.31,2017-04-28


In [206]:
temp_merge['ret'].isna().sum()

np.int64(2962)

In [170]:
file = "s3://buc-vin0358/temp_merge.csv"
temp_merge.to_csv(file)

In [207]:
temp_merge.columns

Index(['Unnamed: 0', 'isin', 'cusip', 'permno', 'namedt', 'nameenddt', 'shrcd',
       'exchcd', 'siccd', 'ncusip', 'ticker', 'comnam', 'shrcls', 'permco_x',
       'hexcd', 'st_date', 'end_date', 'namedum', 'date_x', 'year', 'month',
       'gvkey', 'datadate', 'at', 'ceq', 'sale', 'ni', 'permco_y', 'linktype',
       'linkprim', 'linkdt', 'linkenddt', 'date_y', 'ret', 'shrout', 'prc',
       'date_dt'],
      dtype='object')

In [173]:
len(temp_merge)

174300

In [208]:
file = "s3://buc-vin0358/US_trucost.csv"
US_trucost = pd.read_csv(file)

In [209]:
US_trucost.head()

,Unnamed: 0,TCUID,Company,ISIN,FinancialYear,GICS_Sector_Code,GICS_Sector_Name,GICS_Industry_Group_Code,GICS_Industry_Group_Name,GICS_Industry_Code,...,TotalDirect(USDmn),TotalIndirect(USDmn),Total_DirectAndIndirect(USDmn),GHG_Direct(USDmn),GHG_Indirect(USDmn),GHG_Total(USDmn),GHG_DirectImpactRatio,GHG_IndirectImpactRatio,GHG_TotalImpactRatio,EffectiveDate
0,360,42729,Tyco International PLC,IE00BQRQXQ92,2005,20.0,Industrials,2010.0,Capital Goods,201050.0,...,53.388483,527.068493,580.456976,25.413463,240.684383,266.097846,0.063337,0.599847,0.663184,2009-05-15 12:48:46.67
1,361,42729,Tyco International PLC,IE00BQRQXQ92,2006,20.0,Industrials,2010.0,Capital Goods,201050.0,...,53.412588,498.142054,551.554642,25.368609,225.563068,250.931677,0.061935,0.550691,0.612626,2009-05-15 12:48:46.67
2,362,42729,Tyco International PLC,IE00BQRQXQ92,2007,20.0,Industrials,2010.0,Capital Goods,201050.0,...,24.171680,226.126716,250.298397,12.455727,108.797466,121.253194,0.066321,0.579295,0.645616,2009-05-15 12:48:46.67
3,363,42729,Tyco International PLC,IE00BQRQXQ92,2008,20.0,Industrials,2010.0,Capital Goods,201050.0,...,26.583626,229.062354,255.645980,13.618574,113.313961,126.932535,0.067422,0.560988,0.628410,2011-08-03 12:39:50.92
4,364,42729,Tyco International PLC,IE00BQRQXQ92,2009,20.0,Industrials,2010.0,Capital Goods,201050.0,...,21.645412,200.234121,221.879533,13.918476,95.823620,109.742096,0.080748,0.555918,0.636666,2012-02-08 05:27:43.42


In [210]:
# merge trucost data and wrds data
tru_wrds = temp_merge.merge(US_trucost, left_on=['isin', 'year'], right_on=['ISIN', 'FinancialYear'], how='left')

In [211]:
tru_wrds.head()

,Unnamed: 0_x,isin,cusip,permno,namedt,nameenddt,shrcd,exchcd,siccd,ncusip,...,TotalDirect(USDmn),TotalIndirect(USDmn),Total_DirectAndIndirect(USDmn),GHG_Direct(USDmn),GHG_Indirect(USDmn),GHG_Total(USDmn),GHG_DirectImpactRatio,GHG_IndirectImpactRatio,GHG_TotalImpactRatio,EffectiveDate
0,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,G0684D10,...,0.156354,10.208630,10.364983,0.083470,4.210472,4.293942,0.002032,0.102519,0.104552,2017-10-12 11:54:40.437
1,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,G0684D10,...,0.333687,23.193159,23.526846,0.178141,9.654000,9.832141,0.002034,0.110205,0.112239,2018-10-26 10:14:40.65
2,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,G0684D10,...,0.333687,23.193159,23.526846,0.178141,9.654000,9.832141,0.002034,0.110205,0.112239,2018-10-26 10:14:40.65
3,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,G0684D10,...,0.333687,23.193159,23.526846,0.178141,9.654000,9.832141,0.002034,0.110205,0.112239,2018-10-26 10:14:40.65
4,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,G0684D10,...,0.333687,23.193159,23.526846,0.178141,9.654000,9.832141,0.002034,0.110205,0.112239,2018-10-26 10:14:40.65


In [214]:
file = "s3://buc-vin0358/US_data/US_tru_wrds.csv"
tru_wrds.to_csv(file)

In [2]:
file = "s3://buc-vin0358/US_data/US_tru_wrds.csv"
tru_wrds = pd.read_csv(file)

C:\Users\jiangao\AppData\Local\Temp\ipykernel_17400\1153908210.py:2: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  tru_wrds = pd.read_csv(file)


In [4]:
tru_wrds.columns

Index(['Unnamed: 0', 'Unnamed: 0_x', 'isin', 'cusip', 'permno', 'namedt',
       'nameenddt', 'shrcd', 'exchcd', 'siccd', 'ncusip', 'ticker', 'comnam',
       'shrcls', 'permco_x', 'hexcd', 'st_date', 'end_date', 'namedum',
       'date_x', 'year', 'month', 'gvkey', 'datadate', 'at', 'ceq', 'sale',
       'ni', 'permco_y', 'linktype', 'linkprim', 'linkdt', 'linkenddt',
       'date_y', 'ret', 'shrout', 'prc', 'date_dt', 'Unnamed: 0_y', 'TCUID',
       'Company', 'ISIN', 'FinancialYear', 'GICS_Sector_Code',
       'GICS_Sector_Name', 'GICS_Industry_Group_Code',
       'GICS_Industry_Group_Name', 'GICS_Industry_Code', 'GICS_Industry_Name',
       'GICS_Sub_Industry_Code', 'GICS_Sub_Industry_Name', 'GICS Description',
       'Country', 'CarbonScope1(CO2e)', 'CarbonScope2(CO2e)',
       'CarbonScope3(CO2e)', 'Carbon_FirstTierIndirect(CO2e)',
       'Carbon_DirectAndFirstTierIndirect(CO2e)',
       'CarbonIntensityScope1(CO2e/USDmn)',
       'CarbonIntensityScope2(CO2e/USDmn)',
       'Carb

In [5]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [10]:
results = smf.ols('ret ~ GHG_TotalImpactRatio + at + sale + ni', data = tru_wrds).fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                    ret   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     2.016
Date:                Tue, 10 Mar 2026   Prob (F-statistic):             0.0893
Time:                        13:51:27   Log-Likelihood:                 31942.
No. Observations:               66572   AIC:                        -6.387e+04
Df Residuals:                   66567   BIC:                        -6.383e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                0.0143 

In [9]:
tru_wrds['CarbonIntensityScope1(CO2e/USDmn)'].describe()

count    67408.000000
mean       105.468396
std        458.233973
min          0.000000
25%          3.981511
50%         12.880980
75%         23.088056
max      14227.623824
Name: CarbonIntensityScope1(CO2e/USDmn), dtype: float64

In [12]:
file = "s3://buc-vin0358/US_trucost.csv"
US_tru = pd.read_csv(file)

In [14]:
US_tru.columns

Index(['Unnamed: 0', 'TCUID', 'Company', 'ISIN', 'FinancialYear',
       'GICS_Sector_Code', 'GICS_Sector_Name', 'GICS_Industry_Group_Code',
       'GICS_Industry_Group_Name', 'GICS_Industry_Code', 'GICS_Industry_Name',
       'GICS_Sub_Industry_Code', 'GICS_Sub_Industry_Name', 'GICS Description',
       'Country', 'CarbonScope1(CO2e)', 'CarbonScope2(CO2e)',
       'CarbonScope3(CO2e)', 'Carbon_FirstTierIndirect(CO2e)',
       'Carbon_DirectAndFirstTierIndirect(CO2e)',
       'CarbonIntensityScope1(CO2e/USDmn)',
       'CarbonIntensityScope2(CO2e/USDmn)',
       'CarbonIntensityScope3(CO2e/USDmn)',
       'CarbonIntensityDirect(CO2e/USDmn)',
       'CarbonIntensityFirstTierIndirect(CO2e/USDmn)',
       'CarbonIntensity_DirectAndFirstTierIndirect(CO2e/USDmn)',
       'TotalDirect(USDmn)', 'TotalIndirect(USDmn)',
       'Total_DirectAndIndirect(USDmn)', 'GHG_Direct(USDmn)',
       'GHG_Indirect(USDmn)', 'GHG_Total(USDmn)', 'GHG_DirectImpactRatio',
       'GHG_IndirectImpactRatio', 'GHG_T

In [15]:
file = "s3://buc-vin0358/trucost_use_sorted_countryIndustry.csv"
US_tru_sorted = pd.read_csv(file)

In [16]:
US_tru_sorted.head()

,TCUID,Company,ISIN,FinancialYear,GICS_Sector_Code,GICS_Sector_Name,GICS_Industry_Group_Code,GICS_Industry_Group_Name,GICS_Industry_Code,GICS_Industry_Name,...,TotalDirect(USDmn),TotalIndirect(USDmn),Total_DirectAndIndirect(USDmn),GHG_Direct(USDmn),GHG_Indirect(USDmn),GHG_Total(USDmn),GHG_DirectImpactRatio,GHG_IndirectImpactRatio,GHG_TotalImpactRatio,EffectiveDate
0,53242,Banco BBVA Argentina S.A.,ARP125991090,2005,40.0,Financials,4010.0,Banks,401010.0,Banks,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,NaN,4.0,2009-05-15 12:48:46.67
1,53623,Banco Macro S.A.,ARBANS010010,2005,40.0,Financials,4010.0,Banks,401010.0,Banks,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,2009-05-15 12:48:46.67
2,238954,Telecom Argentina S.A.,US8792732096,2005,50.0,Communication Services,5010.0,Telecommunication Services,501010.0,Diversified Telecommunication Services,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009-05-15 12:48:46.67
3,53609,Ternium Argentina S.A.,ARSIDE010029,2005,15.0,Materials,1510.0,Materials,151040.0,Metals & Mining,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,0.0,4.0,2009-05-15 12:48:46.67
4,67944,Aluar Aluminio Argentino S.A.I.C.,ARALUA010258,2005,15.0,Materials,1510.0,Materials,151040.0,Metals & Mining,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,2009-05-15 12:48:46.67


In [28]:
temp = US_tru_sorted[US_tru_sorted['Country'] == 'UNITED STATES']
temp.head()

,TCUID,Company,ISIN,FinancialYear,GICS_Sector_Code,GICS_Sector_Name,GICS_Industry_Group_Code,GICS_Industry_Group_Name,GICS_Industry_Code,GICS_Industry_Name,...,TotalDirect(USDmn),TotalIndirect(USDmn),Total_DirectAndIndirect(USDmn),GHG_Direct(USDmn),GHG_Indirect(USDmn),GHG_Total(USDmn),GHG_DirectImpactRatio,GHG_IndirectImpactRatio,GHG_TotalImpactRatio,EffectiveDate
2842,43887,ITT Inc.,US45073V1089,2005,20.0,Industrials,2010.0,Capital Goods,201010.0,Aerospace & Defense,...,1.0,2.0,2.0,1.0,2.0,2.0,1.0,3.0,3.0,2009-05-15 12:48:46.67
2843,43968,Northrop Grumman Corporation,US6668071029,2005,20.0,Industrials,2010.0,Capital Goods,201010.0,Aerospace & Defense,...,3.0,3.0,3.0,3.0,3.0,3.0,1.0,1.0,1.0,2009-05-15 12:48:46.67
2844,44008,Raytheon Company,US7551115071,2005,20.0,Industrials,2010.0,Capital Goods,201010.0,Aerospace & Defense,...,2.0,2.0,2.0,2.0,2.0,2.0,1.0,0.0,0.0,2009-05-15 12:48:46.67
2845,44178,The Boeing Company,US0970231058,2005,20.0,Industrials,2010.0,Capital Goods,201010.0,Aerospace & Defense,...,4.0,4.0,4.0,4.0,4.0,4.0,2.0,1.0,2.0,2009-05-15 12:48:46.67
2846,44209,General Dynamics Corporation,US3695501086,2005,20.0,Industrials,2010.0,Capital Goods,201010.0,Aerospace & Defense,...,4.0,3.0,3.0,4.0,3.0,3.0,4.0,3.0,3.0,2009-05-26 13:00:57.113


In [29]:
isin_cusip_expand = pd.read_csv("s3://buc-vin0358/US_isin_cusip_permno_ticker_expand.csv")
merged_use = pd.read_csv("s3://buc-vin0358/US_wrds_crsp_comp.csv")

temp_merge = isin_cusip_expand.merge(merged_use, left_on=['permno', 'year', 'month'], right_on = ['permno', 'year', 'month'], how='left')

tru_sorted = US_tru_sorted

US_tru_sorted = tru_sorted[tru_sorted['Country'] == 'UNITED STATES']

US_tru_wrds = temp_merge.merge(US_tru_sorted, left_on=['isin', 'year'], right_on=['ISIN', 'FinancialYear'], how='left')

In [36]:
US_tru_wrds.columns

Index(['Unnamed: 0.1', 'Unnamed: 0_x', 'isin', 'cusip', 'permno', 'namedt',
       'nameenddt', 'shrcd', 'exchcd', 'siccd', 'ncusip', 'ticker', 'comnam',
       'shrcls', 'permco_x', 'hexcd', 'st_date', 'end_date', 'namedum',
       'date_x', 'year', 'month', 'Unnamed: 0_y', 'gvkey', 'datadate', 'at',
       'ceq', 'sale', 'ni', 'permco_y', 'linktype', 'linkprim', 'linkdt',
       'linkenddt', 'date_y', 'ret', 'shrout', 'prc', 'date_dt', 'TCUID',
       'Company', 'ISIN', 'FinancialYear', 'GICS_Sector_Code',
       'GICS_Sector_Name', 'GICS_Industry_Group_Code',
       'GICS_Industry_Group_Name', 'GICS_Industry_Code', 'GICS_Industry_Name',
       'GICS_Sub_Industry_Code', 'GICS_Sub_Industry_Name', 'GICS Description',
       'Country', 'CarbonScope1(CO2e)', 'CarbonScope2(CO2e)',
       'CarbonScope3(CO2e)', 'Carbon_FirstTierIndirect(CO2e)',
       'Carbon_DirectAndFirstTierIndirect(CO2e)',
       'CarbonIntensityScope1(CO2e/USDmn)',
       'CarbonIntensityScope2(CO2e/USDmn)',
       'Ca

In [37]:
US_tru_wrds_sorted = US_tru_wrds

In [38]:
US_tru_wrds_sorted.head()

,Unnamed: 0.1,Unnamed: 0_x,isin,cusip,permno,namedt,nameenddt,shrcd,exchcd,siccd,...,TotalDirect(USDmn),TotalIndirect(USDmn),Total_DirectAndIndirect(USDmn),GHG_Direct(USDmn),GHG_Indirect(USDmn),GHG_Total(USDmn),GHG_DirectImpactRatio,GHG_IndirectImpactRatio,GHG_TotalImpactRatio,EffectiveDate
0,10150,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,...,3.0,3.0,3.0,3.0,3.0,3.0,1.0,1.0,1.0,2017-10-12 11:54:40.437
1,10150,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,...,4.0,3.0,3.0,3.0,3.0,3.0,1.0,2.0,1.0,2018-10-26 10:14:40.65
2,10150,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,...,4.0,3.0,3.0,3.0,3.0,3.0,1.0,2.0,1.0,2018-10-26 10:14:40.65
3,10150,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,...,4.0,3.0,3.0,3.0,3.0,3.0,1.0,2.0,1.0,2018-10-26 10:14:40.65
4,10150,10150,BMG0684D1074,G0684D10,16496,2016-12-09,2021-12-31,12.0,1.0,6311.0,...,4.0,3.0,3.0,3.0,3.0,3.0,1.0,2.0,1.0,2018-10-26 10:14:40.65


In [32]:
file = "s3://buc-vin0358/US_data/US_tru_wrds_sorted.csv"
US_tru_wrds_sorted.to_csv(file)

In [40]:
US_tru_wrds_sorted.columns

Index(['Unnamed: 0.1', 'Unnamed: 0_x', 'isin', 'cusip', 'permno', 'namedt',
       'nameenddt', 'shrcd', 'exchcd', 'siccd', 'ncusip', 'ticker', 'comnam',
       'shrcls', 'permco_x', 'hexcd', 'st_date', 'end_date', 'namedum',
       'date_x', 'year', 'month', 'Unnamed: 0_y', 'gvkey', 'datadate', 'at',
       'ceq', 'sale', 'ni', 'permco_y', 'linktype', 'linkprim', 'linkdt',
       'linkenddt', 'date_y', 'ret', 'shrout', 'prc', 'date_dt', 'TCUID',
       'Company', 'ISIN', 'FinancialYear', 'GICS_Sector_Code',
       'GICS_Sector_Name', 'GICS_Industry_Group_Code',
       'GICS_Industry_Group_Name', 'GICS_Industry_Code', 'GICS_Industry_Name',
       'GICS_Sub_Industry_Code', 'GICS_Sub_Industry_Name', 'GICS Description',
       'Country', 'CarbonScope1(CO2e)', 'CarbonScope2(CO2e)',
       'CarbonScope3(CO2e)', 'Carbon_FirstTierIndirect(CO2e)',
       'Carbon_DirectAndFirstTierIndirect(CO2e)',
       'CarbonIntensityScope1(CO2e/USDmn)',
       'CarbonIntensityScope2(CO2e/USDmn)',
       'Ca

In [48]:
data = US_tru_wrds_sorted.copy()
data = data.dropna(subset=['ret', 'CarbonIntensityDirect(CO2e/USDmn)', 'at', 'ni', 'sale'])
y = data['ret']
X1 = data[['CarbonIntensityDirect(CO2e/USDmn)', 'at', 'ni', 'sale']]

In [49]:
x = sm.add_constant(X1)
results = sm.OLS(y, x).fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                    ret   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.032
Date:                Tue, 10 Mar 2026   Prob (F-statistic):              0.389
Time:                        15:01:52   Log-Likelihood:                 31930.
No. Observations:               66560   AIC:                        -6.385e+04
Df Residuals:                   66555   BIC:                        -6.380e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

In [1]:
import sys
print(sys.prefix)

/home/j_gao/QuantProjects/CEO_SC/.venv


In [2]:
import pandas as pd